# Canny Kenar Algılama ve Gradyan Yönü Analizi

Bu çalışma; dijital görüntü işlemede kenar tespiti için altın standart kabul edilen **Canny Kenar Algılama Algoritması**'nı (John F. Canny, 1986) ve OpenCV üzerindeki parametrik davranışlarını incelemektedir.

---

## 1. Canny Algoritmasının 5 Aşaması

Canny algoritması, gürültüyü en aza indirirken kenarları 1 piksel kalınlığında ince hatlar halinde tespit etmek için 5 ardışık adımdan oluşur:

1. **Gaussian Blur ile Gürültü Azaltma:** Görüntü $5 \times 5$ Gaussian çekirdeği ile yumuşatılarak türev alma sırasında oluşabilecek yüksek frekanslı gürültüler bastırılır:
   $$G(x, y) = \frac{1}{2\pi \sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$

2. **Yoğunluk Gradyanı ve Açısı Hesabı:** Sobel yatay ($G_x$) ve dikey ($G_y$) filtreleri ile her pikseldeki eğim büyüklüğü ve yönü bulunur:
   $$G = \sqrt{G_x^2 + G_y^2}, \quad \theta = \arctan\left(\frac{G_y}{G_x}\right)$$
   Bulunan $\theta$ açısı 4 ana doğrultuya yuvarlanır: $0^\circ, 45^\circ, 90^\circ, 135^\circ$.

3. **Maksimum Olmayanların Bastırılması (Non-Maximum Suppression - NMS):** Gradyan yönündeki komşu pikseller kıyaslanır. Piksel, kendi gradyan doğrultusundaki yerel tepe noktası (maximum) değilse değeri sıfırlanır. Bu işlem kenarları tek piksel inceliğine getirir.

4. **Çift Eşikleme (Double Thresholding):**
   - $G \ge \text{maxVal}$ ise **Güçlü Kenar** (Strong Edge).
   - $\text{minVal} \le G < \text{maxVal}$ ise **Zayıf Kenar** (Weak Edge).
   - $G < \text{minVal}$ ise kenar değil (Non-edge), sıfırlanır.

5. **Histerezis ile Kenar İzleme (Edge Tracking by Hysteresis):** Bir zayıf kenar pikseli, 8-komşuluğunda güçlü bir kenara bağlıysa nihai kenar kabul edilir; aksi takdirde gürültü sayılarak atılır.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Görüntüyü Gri Seviye Olarak Yükleme
image = cv2.imread('sample_building.jpg', cv2.IMREAD_GRAYSCALE)

if image is None:
    # Test amaçlı sentetik geometrik desen üretimi
    image = np.zeros((400, 400), dtype=np.uint8)
    cv2.circle(image, (200, 200), 100, 255, -1)
    cv2.rectangle(image, (80, 80), (220, 220), 150, -1)
    # Gürültü ekleme
    noise = np.random.normal(0, 15, image.shape).astype(np.uint8)
    image = cv2.add(image, noise)

print(f"Görüntü Boyutları: {image.shape}, Veri Tipi: {image.dtype}")


## 2. Gaussian Yumuşatma ve Sobel Gradyanı ile Karşılaştırma

In [ ]:
# Ön filtreleme
blurred = cv2.GaussianBlur(image, (5, 5), 1.4)

# Sobel gradyanı
sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel_magnitude = np.hypot(sobelx, sobely)
sobel_normalized = np.uint8(255 * sobel_magnitude / np.max(sobel_magnitude))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title("Orijinal Gri Görüntü")
plt.imshow(image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Klasik Sobel Kenar Gradyanı (NMS Uygulanmamış)")
plt.imshow(sobel_normalized, cmap='gray')
plt.axis('off')
plt.show()


## 3. Canny Kenar Algılama Parametrelerinin İncelenmesi

`cv2.Canny(image, threshold1, threshold2, apertureSize=3, L2gradient=False)` fonksiyonunda:
- `threshold1` (minVal): Zayıf kenar alt sınırı.
- `threshold2` (maxVal): Güçlü kenar üst sınırı.
Genel kural olarak $\frac{\text{threshold2}}{\text{threshold1}}$ oranının $2:1$ veya $3:1$ seçilmesi önerilir.


In [ ]:
# Farklı eşik kombinasyonlarının analizi
canny_wide = cv2.Canny(blurred, 30, 90)        # Geniş aralık, hassas kenarlar
canny_mid = cv2.Canny(blurred, 50, 150)       # Dengeli aralık (Önerilen)
canny_tight = cv2.Canny(blurred, 100, 200)    # Sıkı aralık, sadece belirgin hatlar

# Otomatik Eşik Belirleme (Otsu/Medyan Tabanlı)
median_val = np.median(blurred)
sigma = 0.33
auto_lower = int(max(0, (1.0 - sigma) * median_val))
auto_upper = int(min(255, (1.0 + sigma) * median_val))
canny_auto = cv2.Canny(blurred, auto_lower, auto_upper)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(canny_wide, cmap='gray')
axes[0, 0].set_title("Hassas Eşik: min=30, max=90")
axes[0, 0].axis('off')

axes[0, 1].imshow(canny_mid, cmap='gray')
axes[0, 1].set_title("Dengeli Eşik: min=50, max=150")
axes[0, 1].axis('off')

axes[1, 0].imshow(canny_tight, cmap='gray')
axes[1, 0].set_title("Sıkı Eşik: min=100, max=200")
axes[1, 0].axis('off')

axes[1, 1].imshow(canny_auto, cmap='gray')
axes[1, 1].set_title(f"Otomatik Medyan Eşik: min={auto_lower}, max={auto_upper}")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


## 4. Sonuç ve Mühendislik Çıkarımları

1. **Gürültüye Karşı Dayanıklılık:** Gaussian filtresi olmadan doğrudan Canny uygulandığında arka plan dokuları kenar olarak algılanabilir; bu nedenle yumuşatma adımı zorunludur.
2. **NMS Etkisi:** Sobel türevi kenarları kalın ve bulanık bırakırken, Canny'nin NMS adımı kenarları tek piksel kalınlığa indirger.
3. **Histerezis:** Düşük eşiğin üst eşikle dengelenmesi, kopuk kenar hatlarının sürekliliğini garanti eder.
